# EcoSignal: weekly satellite landscape change
Run cells in order on **Colab → Python 3 → CPU runtime**. No GPU needed.

Uses the authenticated Copernicus Data Space Sentinel Hub Catalog and Process APIs, not Earth Engine. Create a CDSE account at https://dataspace.copernicus.eu/ and create an OAuth client in the Sentinel Hub dashboard/user settings. Confirm API access/processing allowance. Credentials are entered privately and are never exported.

Upload your **exact geomorphology study boundary** as WGS84 GeoJSON. A greater-Raleigh bounding box is not guessed. Convert a shapefile to EPSG:4326 GeoJSON first if necessary. A smoke test uses a small portion of that boundary; full-region mode uses the entire uploaded polygon.

Outputs: weekly NDVI/NDMI/BSI/MNDWI rasters, per-index observation counts, adjacent-week differences, provenance manifest, summary CSV and interactive preview. These are screening indicators, not calibrated erosion/pollution diagnoses. NDMI is vegetation moisture context, not volumetric soil moisture.


In [ ]:
%pip install -q requests numpy pandas rasterio scipy shapely pyproj folium matplotlib

In [ ]:
from google.colab import drive, files
from pathlib import Path
from getpass import getpass
import datetime as dt
# Persist cached acquisitions and outputs across runtime resets.
drive.mount('/content/drive')
OUTPUT_BASE = Path('/content/drive/MyDrive/EcoSignal/satellite_weekly')
print('Upload the WGS84 GeoJSON boundary from your geomorphology study.')
uploaded = files.upload()
geojsons = [name for name in uploaded if name.lower().endswith(('.geojson','.json'))]
assert len(geojsons) == 1, 'Upload exactly one GeoJSON boundary file.'
AOI_PATH = geojsons[0]


## Configure periods and first-run scope
Monday-to-Monday periods; `END` is exclusive. Defaults cover the two latest completed weeks in UTC. Cloudy weeks can have no comparable pixels—this is an honest result, not a processing failure.

**Keep SMOKE_TEST=True on the first run.** This processes roughly a 10×10 km central subset, explicitly labelled as a test. After it succeeds, set False and rerun configuration, catalog check and processing for your full boundary. If the centre misses an irregular boundary, provide a point inside the boundary.


In [ ]:
today = dt.datetime.now(dt.timezone.utc).date()
END = today - dt.timedelta(days=today.weekday())
START = END - dt.timedelta(weeks=2)
# For historical dates, replace with: START=dt.date(2026,9,21); END=dt.date(2026,10,5)
SMOKE_TEST = True
MIN_CLEAR_OBSERVATIONS = 1  # Increase to 2 for stricter coverage; many weekly gaps may result.
MAX_REQUESTS = 250         # Safety cap, not a guarantee of free processing.
SMOKE_CENTRE_LON_LAT = None # Optional (longitude, latitude), inside your boundary.
CLIENT_ID = getpass('CDSE OAuth client ID: ')
CLIENT_SECRET = getpass('CDSE OAuth client secret: ')


## Pipeline definitions

In [ ]:
"""EcoSignal weekly satellite context. No API credentials are saved."""
import datetime as dt
import hashlib
import json
import math
import time
import warnings
from pathlib import Path
import numpy as np
import requests
import rasterio
from rasterio.features import geometry_mask
from rasterio.transform import from_origin
from rasterio.warp import transform_bounds
from scipy.ndimage import binary_dilation
from shapely.geometry import shape, mapping, box
from shapely.ops import unary_union, transform
from pyproj import Transformer

VERSION = '1.0.0'
NAMES = ['NDVI', 'NDMI', 'BSI', 'MNDWI']
BASE = 'https://sh.dataspace.copernicus.eu'
EVALSCRIPT = '''//VERSION=3
function setup() {
 return {input:[{bands:["B02","B03","B04","B08","B11","SCL","CLD","dataMask"],
 units:["REFLECTANCE","REFLECTANCE","REFLECTANCE","REFLECTANCE","REFLECTANCE","DN","PERCENT","DN"]}],
 output:{bands:8,sampleType:"FLOAT32"}};
}
function evaluatePixel(s) {return [s.B02,s.B03,s.B04,s.B08,s.B11,s.SCL,s.CLD,s.dataMask];}
'''

class CDSE:
    def __init__(self, client_id, client_secret):
        self.client_id, self.client_secret = client_id, client_secret
        self.session = requests.Session()
        self.expiry = 0
    def authenticate(self):
        r = self.session.post('https://identity.dataspace.copernicus.eu/auth/realms/CDSE/protocol/openid-connect/token',
            data={'grant_type':'client_credentials','client_id':self.client_id,'client_secret':self.client_secret}, timeout=60)
        if not r.ok:
            raise RuntimeError(f'Authentication failed ({r.status_code}). Check your CDSE OAuth client and account access.')
        token = r.json()
        self.session.headers['Authorization'] = 'Bearer ' + token['access_token']
        self.expiry = time.time() + token.get('expires_in',300) - 30
    def post(self, path, payload, binary=False):
        for attempt in range(5):
            if time.time() >= self.expiry:
                self.authenticate()
            r = self.session.post(BASE+path, json=payload, timeout=180)
            if r.status_code == 401:
                self.expiry = 0
                continue
            if r.status_code == 429 or r.status_code >= 500:
                if attempt < 4:
                    time.sleep(min(2**attempt * 2, 20))
                    continue
            if not r.ok:
                # Never print request headers, tokens, or credentials.
                raise RuntimeError(f'CDSE {path}: HTTP {r.status_code}. ' +
                    ('Check your processing allowance/access.' if r.status_code in (403,429) else r.text[:500]))
            return r.content if binary else r.json()
        raise RuntimeError('API retry limit reached. Rerun later; cached acquisitions are retained.')
    def catalog(self, bbox, start, end):
        payload = {'bbox':list(bbox),'datetime':f'{start}T00:00:00Z/{end}T00:00:00Z',
                   'collections':['sentinel-2-l2a'],'limit':100}
        features, seen = [], set()
        while True:
            result = self.post('/catalog/v1/search',payload)
            features.extend(result.get('features',[]))
            nxt = result.get('context',{}).get('next')
            if nxt is None: break
            if str(nxt) in seen: raise RuntimeError('Catalog pagination repeated a token.')
            seen.add(str(nxt)); payload['next'] = nxt
        return features
    def image(self, bounds, width, height, date):
        day = dt.date.fromisoformat(date)
        payload = {'input':{'bounds':{'bbox':list(bounds), 'properties':{'crs':'http://www.opengis.net/def/crs/EPSG/0/32617'}},
            'data':[{'type':'sentinel-2-l2a','dataFilter':{'timeRange':{
                'from':date+'T00:00:00Z','to':str(day+dt.timedelta(days=1))+'T00:00:00Z'},
                'mosaickingOrder':'mostRecent','maxCloudCoverage':100},
                'processing':{'upsampling':'NEAREST','downsampling':'NEAREST','harmonizeValues':True}}]},
            'output':{'width':width,'height':height,'responses':[{'identifier':'default','format':{'type':'image/tiff'}}]},
            'evalscript':EVALSCRIPT}
        return self.post('/process/v1',payload,binary=True)

def load_aoi(path):
    obj = json.loads(Path(path).read_text())
    if obj.get('crs'):
        raise ValueError('Export the AOI as RFC7946 GeoJSON in WGS84 (EPSG:4326), without a legacy crs field.')
    if obj['type']=='FeatureCollection':
        g = unary_union([shape(f['geometry']) for f in obj['features'] if f.get('geometry')])
    elif obj['type']=='Feature': g = shape(obj['geometry'])
    else: g = shape(obj)
    if g.is_empty or not g.is_valid or g.geom_type not in ('Polygon','MultiPolygon'):
        raise ValueError('AOI must be a valid nonempty polygon/multipolygon.')
    west,south,east,north = g.bounds
    if not (-80.5 < west < east < -77 and 34 < south < north < 37.5):
        raise ValueError('Boundary is outside the expected Raleigh region or has incorrect coordinates/CRS.')
    return g

def week_starts(start,end):
    a,b = dt.date.fromisoformat(start),dt.date.fromisoformat(end)
    if a.weekday() != 0 or b.weekday() != 0 or b <= a or (b-a).days % 7:
        raise ValueError('Start/end must be Mondays; end is exclusive, after start.')
    return [a+dt.timedelta(days=n) for n in range(0,(b-a).days,7)]

def tiles_for(g, resolution=20, tile_pixels=1024):
    if resolution != 20: raise ValueError('This version uses a fixed 20 m analysis grid.')
    pg = transform(Transformer.from_crs(4326,32617,always_xy=True).transform,g)
    left,bottom,right,top = pg.bounds
    left=math.floor(left/resolution)*resolution; top=math.ceil(top/resolution)*resolution
    w=math.ceil((right-left)/resolution); h=math.ceil((top-bottom)/resolution)
    tiles=[]
    for row in range(0,h,tile_pixels):
        for col in range(0,w,tile_pixels):
            tw,th=min(tile_pixels,w-col),min(tile_pixels,h-row)
            x,y=left+col*resolution,top-row*resolution
            bounds=(x,y-th*resolution,x+tw*resolution,y)
            if not pg.intersects(box(*bounds)): continue
            affine=from_origin(x,y,resolution,resolution)
            mask=geometry_mask([mapping(pg)],(th,tw),affine,invert=True)
            if not mask.any(): continue
            tiles.append({'id':f'r{row}_c{col}','bounds':bounds,'width':tw,'height':th,'transform':affine,'mask':mask})
    return tiles

def ratio(a,b):
    out=np.full(a.shape,np.nan,dtype=np.float32)
    np.divide(a-b,a+b,out=out,where=np.abs(a+b)>1e-6)
    return out

def indices(raw, aoi_mask, cloud_probability=40, buffer_pixels=2):
    blue,green,red,nir,swir,scl,cld,data=raw
    bad=(~np.isin(scl,[4,5,6])) | (cld>=cloud_probability) | (data<.5)
    if buffer_pixels: bad=binary_dilation(bad,iterations=buffer_pixels)
    clear=(~bad)&aoi_mask&np.isfinite(raw[:5]).all(axis=0)
    ndvi=ratio(nir,red); ndmi=ratio(nir,swir)
    bsi=ratio(swir+red,nir+blue); mndwi=ratio(green,swir)
    # Conservative water exclusion; threshold is adjustable screening logic, not ground truth.
    water=(scl==6)|(mndwi>0.0)
    values=np.stack([ndvi,ndmi,bsi,mndwi]).astype(np.float32)
    values[:,~clear]=np.nan
    values[:3,water]=np.nan
    # NDMI is interpreted only where vegetation is present.
    values[1,ndvi<.2]=np.nan
    return values,clear

def composite(observations, shape_, min_obs=1):
    if not observations:
        return np.full((4,*shape_),np.nan,dtype=np.float32),np.zeros((4,*shape_),dtype=np.uint16)
    stack=np.stack(observations)
    counts=np.isfinite(stack).sum(axis=0).astype(np.uint16)
    with warnings.catch_warnings():
        warnings.simplefilter('ignore',RuntimeWarning)
        med=np.nanmedian(stack,axis=0).astype(np.float32)
    med[counts<min_obs]=np.nan
    return med,counts

def difference(previous,current):
    valid=np.isfinite(previous)&np.isfinite(current)
    out=np.full(previous.shape,np.nan,dtype=np.float32)
    np.subtract(current,previous,out=out,where=valid)
    return out

def write_tif(path, array, tile, names, tags):
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    with rasterio.open(path,'w',driver='GTiff',height=tile['height'],width=tile['width'],
        count=len(array),dtype='float32',crs='EPSG:32617',transform=tile['transform'],
        nodata=np.nan,compress='deflate',tiled=True) as ds:
        ds.write(array.astype(np.float32));ds.descriptions=tuple(names)
        ds.update_tags(**{k:str(v) for k,v in tags.items()})

def run(client, geometry, start, end, output, min_obs=1, max_requests=250):
    root=Path(output);root.mkdir(parents=True,exist_ok=True)
    weeks=week_starts(start,end);tiles=tiles_for(geometry)
    scenes=client.catalog(geometry.bounds,start,end)
    dates=sorted({f['properties']['datetime'][:10] for f in scenes
                  if start<=f['properties']['datetime'][:10]<end})
    signature=hashlib.sha256((json.dumps(mapping(geometry),sort_keys=True)+EVALSCRIPT+VERSION).encode()).hexdigest()[:12]
    cache=root/'cache'/signature;cache.mkdir(parents=True,exist_ok=True)
    requests_needed=sum(not (cache/f"{t['id']}_{d}.tif").exists() for t in tiles for d in dates)
    print(f'{len(tiles)} tiles; {len(dates)} acquisition days; {requests_needed} uncached processing requests.')
    if requests_needed>max_requests:
        raise RuntimeError('Request budget exceeded. First shorten the date range or use a SMALLER boundary for a smoke test, then increase MAX_REQUESTS deliberately.')
    if not dates: raise RuntimeError('No catalog acquisitions in the date range. Check dates, AOI and account.')
    (root/'catalog.json').write_text(json.dumps(scenes,indent=2))
    (root/'study_area.geojson').write_text(json.dumps(mapping(geometry)))
    manifest={'version':VERSION,'source':'Copernicus Sentinel-2 L2A','start':start,'end_exclusive':end,
        'crs':'EPSG:32617','resolution_m':20,'indices':NAMES,'min_clear_observations':min_obs,
        'cloud_probability_threshold_percent':40,'cloud_buffer_m':40,
        'acquisition_dates':dates,'outputs':[],
        'limitations':['Weekly values are median per-day indices, not daily measurements.',
        'NDMI is a vegetation moisture indicator, not soil water percentage.',
        'BSI and water thresholds require local validation; changes do not establish causes.',
        'Seasonality, agriculture, construction and residual cloud/shadow can affect changes.',
        'Per-day mosaics may include multiple passes/tiles; catalog.json records candidate scenes.',
        'Only adjacent fixed weeks are compared; missing weeks are not silently bridged.',
        'A one-observation week is sparse evidence, not calibrated confidence.']}
    summaries={}
    for tile_n,tile in enumerate(tiles,1):
        previous=None
        for week in weeks:
            wd=[d for d in dates if week.isoformat()<=d<(week+dt.timedelta(days=7)).isoformat()]
            observations=[]
            for d in wd:
                p=cache/f"{tile['id']}_{d}.tif"
                if not p.exists():
                    blob=client.image(tile['bounds'],tile['width'],tile['height'],d)
                    # Validate bytes before caching; never cache an API error as an image.
                    with rasterio.io.MemoryFile(blob) as mem:
                        with mem.open() as ds:
                            if ds.count!=8 or ds.shape!=(tile['height'],tile['width']):
                                raise RuntimeError('Unexpected processing response dimensions/bands.')
                    p.write_bytes(blob)
                with rasterio.open(p) as ds: raw=ds.read()
                values,_=indices(raw,tile['mask']);observations.append(values)
            med,counts=composite(observations,(tile['height'],tile['width']),min_obs)
            tags={'week_start':week,'end_exclusive':week+dt.timedelta(days=7),'dates':','.join(wd),'version':VERSION}
            rel=f"weekly/{week}/{tile['id']}.tif"
            write_tif(root/rel,np.concatenate([med,counts.astype(np.float32)]),tile,NAMES+[n+'_clear_count' for n in NAMES],tags)
            manifest['outputs'].append({'kind':'weekly','week':str(week),'tile':tile['id'],'file':rel,
                'bbox_wgs84':list(transform_bounds(32617,4326,*tile['bounds'])),'acquisition_dates':wd})
            for j,name in enumerate(NAMES):
                key=(str(week),name);s=summaries.setdefault(key,{'sum':0.,'valid_pixels':0,'aoi_pixels':0,'delta_sum':0.,'comparison_pixels':0})
                finite=np.isfinite(med[j]);s['sum']+=float(np.nansum(med[j],dtype=np.float64))
                s['valid_pixels']+=int(finite.sum());s['aoi_pixels']+=int(tile['mask'].sum())
            if previous is not None:
                delta=difference(previous,med)
                rel=f"changes/{week}/{tile['id']}.tif"
                write_tif(root/rel,delta,tile,['delta_'+n for n in NAMES],tags)
                manifest['outputs'].append({'kind':'change','week':str(week),'previous_week':str(week-dt.timedelta(days=7)),
                    'tile':tile['id'],'file':rel,'bbox_wgs84':list(transform_bounds(32617,4326,*tile['bounds']))})
                for j,name in enumerate(NAMES):
                    s=summaries[(str(week),name)];s['delta_sum']+=float(np.nansum(delta[j],dtype=np.float64))
                    s['comparison_pixels']+=int(np.isfinite(delta[j]).sum())
            previous=med
            (root/'manifest.json').write_text(json.dumps(manifest,indent=2))
            print(f'Tile {tile_n}/{len(tiles)}, week {week}: {len(wd)} acquisition days processed')
    import pandas as pd
    rows=[]
    for (week,name),s in sorted(summaries.items()):
        rows.append({'week':week,'index':name,'mean':s['sum']/s['valid_pixels'] if s['valid_pixels'] else None,
            'valid_area_fraction':s['valid_pixels']/s['aoi_pixels'],'comparison_area_fraction':s['comparison_pixels']/s['aoi_pixels'],
            'mean_change_on_common_pixels':s['delta_sum']/s['comparison_pixels'] if s['comparison_pixels'] else None})
    pd.DataFrame(rows).to_csv(root/'weekly_summary.csv',index=False)
    manifest['completed']=True;(root/'manifest.json').write_text(json.dumps(manifest,indent=2))
    return pd.DataFrame(rows),manifest

def preview(root, manifest):
    import folium
    from rasterio.warp import reproject, calculate_default_transform, Resampling
    import matplotlib
    root=Path(root);g=load_aoi(root/'study_area.geojson')
    m=folium.Map(location=[g.centroid.y,g.centroid.x],zoom_start=10,tiles='OpenStreetMap')
    latest=max(o['week'] for o in manifest['outputs'] if o['kind']=='change')
    groups=[folium.FeatureGroup(name='Weekly '+n+' change',show=(j==0)) for j,n in enumerate(NAMES)]
    for entry in manifest['outputs']:
        if entry['kind']!='change' or entry['week']!=latest: continue
        with rasterio.open(root/entry['file']) as ds:
            affine,w,h=calculate_default_transform(ds.crs,3857,ds.width,ds.height,*ds.bounds)
            for j,group in enumerate(groups):
                dest=np.full((h,w),np.nan,dtype=np.float32)
                reproject(source=rasterio.band(ds,j+1),destination=dest,src_transform=ds.transform,
                    src_crs=ds.crs,dst_transform=affine,dst_crs=3857,src_nodata=np.nan,dst_nodata=np.nan,resampling=Resampling.nearest)
                rgba=(matplotlib.colormaps['RdYlGn'](np.clip((np.nan_to_num(dest)+.3)/.6,0,1))*255).astype(np.uint8)
                rgba[:,:,3]=np.where(np.isfinite(dest),170,0).astype(np.uint8)
                west,south,east,north=transform_bounds(3857,4326,*rasterio.transform.array_bounds(h,w,affine))
                folium.raster_layers.ImageOverlay(rgba,bounds=[[south,west],[north,east]],name=entry['tile']).add_to(group)
    for group in groups: group.add_to(m)
    folium.GeoJson(mapping(g),name='Study boundary').add_to(m)
    folium.LayerControl().add_to(m)
    label=f'Week {latest} minus previous week. Red: decrease; yellow: near zero; green: increase. Display scale −0.3 to +0.3; transparent: no comparison. BSI increase means stronger exposed-ground signal, not improvement.'
    m.get_root().html.add_child(folium.Element('<div style="position:fixed;bottom:10px;left:10px;z-index:9999;background:white;padding:8px;max-width:450px">'+label+'</div>'))
    m.save(str(root/'preview.html'));return m

def self_test():
    raw=np.zeros((8,8,8),dtype=np.float32)
    raw[:5]=np.array([.1,.2,.1,.6,.2])[:,None,None];raw[5]=4;raw[7]=1
    mask=np.ones((8,8),dtype=bool)
    v,clear=indices(raw,mask,buffer_pixels=0)
    np.testing.assert_allclose(v[:,3,3],[5/7,.5,-.4,0],atol=1e-6)
    raw[5,3,3]=9
    v,clear=indices(raw,mask,buffer_pixels=1)
    assert not clear[3,3] and not clear[3,4] and np.isnan(v[:,3,3]).all()
    empty,counts=composite([],(8,8));assert np.isnan(empty).all() and not counts.any()
    a=np.ones((4,8,8),dtype=np.float32);b=a*1.2;b[:,0,0]=np.nan
    delta=difference(a,b);assert np.isnan(delta[:,0,0]).all()
    np.testing.assert_allclose(delta[:,1,1],.2,atol=1e-6)
    med,n=composite([a,b],(8,8),2);assert np.isnan(med[:,0,0]).all() and (n[:,0,0]==1).all()
    assert len(week_starts('2026-09-21','2026-10-05'))==2
    print('PASS: formulas, cloud buffering, missing weeks, common-pixel changes, observation counts, weekly dates.')


## Offline checks and exact-area setup

In [ ]:
self_test()
full_aoi = load_aoi(AOI_PATH)
aoi = full_aoi
if SMOKE_TEST:
    pt = full_aoi.representative_point()
    lon, lat = SMOKE_CENTRE_LON_LAT or (pt.x, pt.y)
    x,y = Transformer.from_crs(4326,32617,always_xy=True).transform(lon,lat)
    subset = transform(Transformer.from_crs(32617,4326,always_xy=True).transform,box(x-5000,y-5000,x+5000,y+5000))
    aoi = full_aoi.intersection(subset)
    assert not aoi.is_empty, 'Smoke-test centre must overlap your boundary.'
RUN_NAME = ('smoke_test_' if SMOKE_TEST else 'greater_raleigh_') + str(START) + '_' + str(END)
OUTPUT = OUTPUT_BASE / RUN_NAME
client = CDSE(CLIENT_ID, CLIENT_SECRET)
client.authenticate()
print('Authentication OK. Area bounds:', aoi.bounds)
print('Period:', START, 'to', END, '(end exclusive)')
print('Mode:', 'SMALL SUBSET TEST' if SMOKE_TEST else 'FULL GEOMORPHOLOGY AREA')


## Catalog check: inexpensive metadata query before imagery processing

In [ ]:
import pandas as pd
catalog = client.catalog(aoi.bounds,str(START),str(END))
catalog_rows = [{'id':f['id'],'acquired':f['properties'].get('datetime'),'tile_cloud_percent':f['properties'].get('eo:cloud_cover')} for f in catalog]
display(pd.DataFrame(catalog_rows))
print('Catalog records:',len(catalog),'; acquisition days:',len({r['acquired'][:10] for r in catalog_rows}))
assert catalog, 'No acquisitions found; check dates and the AOI.'
print('Tiles to process:',len(tiles_for(aoi)))


## Process weekly imagery
This cell consumes processing allowance. Requests are tiled and cached. Rerunning the same period reuses cached acquisition TIFFs. If a recent product was incomplete at first retrieval, delete only its cached TIFF and rerun to refresh. No scenes are invented or interpolated across missing weeks.


In [ ]:
summary, manifest = run(client,aoi,str(START),str(END),OUTPUT,MIN_CLEAR_OBSERVATIONS,MAX_REQUESTS)
display(summary)
print('Finished:',OUTPUT)
assert manifest.get('completed') is True
assert all(0 <= x <= 1 for x in summary.valid_area_fraction)
if summary.comparison_area_fraction.max() == 0:
    print('No usable adjacent-week comparison. Try an older period or a longer window in a future version; do not treat missing data as zero change.')


## Map preview and output verification
Toggle the four layers. Colours show signed index changes, not a risk classification. BSI increase has a different meaning from greenness increase. Transparent areas lack a valid comparison. No validated change thresholds are assumed.


In [ ]:
m = preview(OUTPUT,manifest)
display(m)
with rasterio.open(OUTPUT / next(o['file'] for o in manifest['outputs'] if o['kind']=='weekly')) as ds:
    assert ds.crs.to_epsg()==32617 and ds.count==8
    assert abs(ds.transform.a-20)<1e-6 and abs(ds.transform.e+20)<1e-6
    print('PASS: GeoTIFF CRS, 20 m grid, four indices and four count bands.')


## Export for app integration
The ZIP contains processed products only; it excludes cached source imagery and credentials. The map preview embeds its overlays, but needs internet for its basemap. `manifest.json` identifies tile paths, dates, bounds and quality settings. App integration should render GeoTIFFs through a raster tile service or convert them to georeferenced map assets; the ZIP itself is not an API server.

A Colab session does **not** keep running after it closes. Rerun periodically for the prototype; deploy the processing function to a scheduled backend for unattended weekly updates. Same-season comparisons and independently checked sites are still required before making causal or accuracy claims.


In [ ]:
import zipfile
bundle = Path('/content') / (RUN_NAME + '.zip')
with zipfile.ZipFile(bundle,'w',zipfile.ZIP_DEFLATED) as z:
    for p in OUTPUT.rglob('*'):
        if p.is_file() and 'cache' not in p.relative_to(OUTPUT).parts:
            z.write(p,p.relative_to(OUTPUT))
print('Saved outputs:', OUTPUT)
files.download(str(bundle))


## Troubleshooting
- **401:** recreate/check the OAuth client ID/secret; do not use your account password.
- **403:** check Sentinel Hub API access and your CDSE processing allowance.
- **429:** processing/rate quota; stop, shorten the range or resume later. Cache survives.
- **No acquisitions:** use historical dates, confirm boundary and account access.
- **Mostly blank map:** cloud/shadow masks, sparse coverage, water exclusion or vegetation-only NDMI. Inspect counts and natural-colour source imagery; never remove masks solely to make the map look complete.
- **Budget cap:** start with the smoke test; full greater Raleigh may require many tiles and requests.
- **No changes on screen:** stable area or missing comparable observations; inspect CSV and acquisition dates.

Documentation: https://documentation.dataspace.copernicus.eu/APIs/SentinelHub/Catalog.html ; https://documentation.dataspace.copernicus.eu/APIs/SentinelHub/Process.html ; https://documentation.dataspace.copernicus.eu/APIs/SentinelHub/Data/S2L2A.html
